<img src="https://huggingface.co/datasets/FineEnvs/SmolDataEnvs/resolve/main/banner.png" width="100%">

# Train a small model on SmolDataEnvs

**5.5K+ RL tasks for hill-climbing small models in code and data science.**

This notebook is the short path from *what is in this dataset* to *a model trained on it*:

1. look at one task — a real table, a question, a gold answer
2. grade an answer the way the dataset grades it: deterministically, no LLM judge
3. fine-tune a small model on 4,677 verified trajectories

It runs end to end on a free Colab T4 with the 360M default. Swap in a 3B model and the same
cells train something worth keeping.

> The RL loop itself (agent in a sandbox, reward from the task's own verifier) lives in the
> [Harbor suites](https://huggingface.co/datasets/FineEnvs/SmolDataEnvs-harbor-train) and needs
> a sandbox provider, so it is not in this notebook. This is the warm start.


In [ ]:
%pip install -q datasets trl peft transformers trackio huggingface_hub math-verify


## 1. One task, end to end

`SmolDataEnvs` is the flat view: one row per task, no runtime required.


In [ ]:
from datasets import load_dataset

ds = load_dataset("FineEnvs/SmolDataEnvs", split="test")
print(ds)

row = ds[0]
print("\nquestion :", row["question"])
print("answer   :", row["answer"], f"({row['reward_mode']})")
print("source   :", row["kaggle_dataset"], "|", row["difficulty_tier"])


The task is not answerable from the text. The agent has to open the data, which lives in a
bucket on the Hub, and compute the answer.


In [ ]:
# The task's tables live in a Hugging Face **bucket**, not a dataset repo, so they come
# down with the bucket API rather than snapshot_download.
from huggingface_hub import list_bucket_tree, download_bucket_files
import pandas as pd, os

prefix = row["bucket_prefix"].rstrip("/") + "/"
items = [i for i in list_bucket_tree(row["hf_bucket"], prefix=prefix, recursive=True)
         if getattr(i, "type", None) == "file"]
os.makedirs("input", exist_ok=True)
targets = [(i.path, "input/" + i.path.split("/")[-1]) for i in items]
download_bucket_files(row["hf_bucket"], files=targets)

print("files:", [t[1] for t in targets])
pd.read_csv(targets[0][1], low_memory=False).head()


## 2. How it is graded

Every task carries a gold answer and a match mode, and the bundled `grader.py` compares them
through a ladder of checks: exact → numeric with tolerances → list and percent normalisation →
symbolic equivalence. No model sits in the reward path, so the signal cannot drift with a judge's
mood, because there is no judge.


In [ ]:
from huggingface_hub import hf_hub_download
import importlib.util, sys

p = hf_hub_download("FineEnvs/SmolDataEnvs", "grader.py", repo_type="dataset")
spec = importlib.util.spec_from_file_location("grader", p)
grader = importlib.util.module_from_spec(spec)
sys.modules["grader"] = grader          # dataclasses in grader.py need this
spec.loader.exec_module(grader)

def score(pred):
    r = grader.grade(row["answer"], pred, reward_mode=row["reward_mode"],
                     abs_tol=row["atol"], rel_tol=row["rtol"])
    return r.reward, r.method

print("gold answer       ->", score(row["answer"]))     # (1.0, 'exact')
print("wrong answer      ->", score("42"))              # (0.0, 'miss')
print("sloppy whitespace ->", score(f"  {row['answer']}  "))  # (1.0, 'exact')


## 3. Fine-tune on verified trajectories

`SmolDataEnvs-sft` is 4,677 complete agent rollouts that **solved** their task under that grader.
It ships as conversational `messages` plus the `bash` tool schema, which is what TRL expects, so
there is no preprocessing step.


In [ ]:
sft = load_dataset("FineEnvs/SmolDataEnvs-sft", split="train")
print(sft)

ex = sft[0]
print("\nturns:", ex["n_turns"], "| difficulty:", ex["difficulty_tier"], "\n")
for m in ex["messages"][:4]:
    body = m.get("content") or m.get("tool_calls")
    print(f"--- {m['role']}\n{str(body)[:300]}\n")


The cell below trains. `MAX_SAMPLES` keeps this notebook honest about its runtime — raise it (or
drop it) for a real run, and switch `MODEL` to `HuggingFaceTB/SmolLM3-3B` on a bigger GPU.


In [ ]:
import os
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
MAX_SAMPLES = 256          # None for the full 4,677
HUB_MODEL_ID = None        # e.g. "your-username/smoldataenvs-sft-360m"

train = sft.select(range(MAX_SAMPLES)) if MAX_SAMPLES else sft
split = train.train_test_split(test_size=0.05, seed=42)

trainer = SFTTrainer(
    model=MODEL,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05),
    args=SFTConfig(
        output_dir="smoldataenvs-sft",
        num_train_epochs=1,
        max_length=4096,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-5,
        gradient_checkpointing=True,
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=25,
        push_to_hub=bool(HUB_MODEL_ID),
        hub_model_id=HUB_MODEL_ID,
    ),
)
trainer.train()


## 4. Where to go next

**Same run, bigger, on a GPU you do not own** — the notebook's training cell exists as a
single-file script:

```bash
MODEL=HuggingFaceTB/SmolLM3-3B FLAVOR=a10g-large \
  ./scripts/run_on_hf_jobs.sh your-username/smoldataenvs-sft-3b
```

**The actual RL** — the three Harbor suites turn each task into a sandboxed environment with its
own verifier, so an agent can be trained against the reward rather than against demonstrations:

```bash
openenv harbor serve \
  --dataset FineEnvs/SmolDataEnvs-harbor-train,FineEnvs/SmolDataEnvs-harbor-eval \
  --llm-url http://127.0.0.1:8000/v1 --model your-model
```

**Browse the tasks** —
[Harbor Visualiser](https://huggingface.co/spaces/HuggingFaceH4/harbor-visualiser?dataset=FineEnvs/SmolDataEnvs-harbor-train)
· [the collection](https://huggingface.co/collections/FineEnvs/smoldataenvs)
